In [2]:
# 🔧 Configuración del notebook
import warnings
warnings.filterwarnings('ignore')

In [5]:
import pandas as pd
import numpy as np
import os

# --- Configura aquí la ruta según tu entorno ---
ARCHIVO = 'datos_con_clusters.csv'

# Intenta cargar desde distintas ubicaciones comunes
rutas = [
    ARCHIVO,                                          # directorio actual
    f'/content/{ARCHIVO}',                            # Colab raíz
    f'/content/drive/MyDrive/{ARCHIVO}',              # Google Drive
    f'../data/{ARCHIVO}',                             # carpeta data relativa
]

df = None
for ruta in rutas:
    if os.path.exists(ruta):
        df = pd.read_csv(ruta)
        print(f"✅ Archivo cargado desde: {ruta}")
        break

if df is None:
    raise FileNotFoundError(
        f"❌ No se encontró '{ARCHIVO}'. "
        "Sube el archivo al entorno o ajusta la ruta."
    )

✅ Archivo cargado desde: /content/drive/MyDrive/datos_con_clusters.csv


Verificamos si los clusters existen:

In [6]:
CLUSTER_COLS = ['cluster_kmeans', 'cluster_dbscan', 'cluster_hierarchical']

print(f"\n📐 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"📋 Columnas totales: {list(df.columns)}\n")

for col in CLUSTER_COLS:
    if col in df.columns:
        n_clusters = df[col].nunique()
        print(f"  ✅ {col}: {n_clusters} valores únicos → {sorted(df[col].unique())}")
    else:
        print(f"  ❌ {col}: NO encontrada — verifica el archivo")

# Seleccionar el método principal para el resto del análisis
METODO_PRINCIPAL = 'cluster_kmeans'
print(f"\n🎯 Método principal seleccionado: {METODO_PRINCIPAL}")


📐 Dimensiones: 32764 filas × 16 columnas
📋 Columnas totales: ['PERCEPHO', 'P103', 'P104', 'P105A', 'EDUCACION_JEFE', 'DOMINIO', 'NUM_SERVICIOS', 'NUM_EQUIPOS', 'NUM_OCUPADOS', 'EDAD_JEFE', 'PCT_NINOS', 'POBREZA', 'HACINAMIENTO', 'cluster_kmeans', 'cluster_dbscan', 'cluster_hierarchical']

  ✅ cluster_kmeans: 5 valores únicos → [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  ✅ cluster_dbscan: 10 valores únicos → [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
  ✅ cluster_hierarchical: 5 valores únicos → [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

🎯 Método principal seleccionado: cluster_kmeans


#Detectamos Outliers utilizando Isolation Forest

In [12]:
# ============================================
# 🌲 DETECCIÓN DE OUTLIERS - ISOLATION FOREST
# ============================================
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# --- 1. Preparamos los features  ---
# --- Se excluyen las columnas de cluster y POBREZA ---
cols_excluir = ['cluster_kmeans', 'cluster_dbscan', 'cluster_hierarchical', 'POBREZA']
feature_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                if col not in cols_excluir]

print(f"📌 Variables usadas para Isolation Forest: {feature_cols}")

X = df[feature_cols].copy()

# --- 2. Escalar No es obligatorio, pero mejora consistencia) ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- 3. Aplicar Isolation Forest ---
iso_forest = IsolationForest(
    n_estimators=100,      # número de árboles
    contamination='auto',    # % esperado de outliers (ajusta según tu dataset)
    random_state=42
)

df['outlier_iforest'] = iso_forest.fit_predict(X_scaled)
# Resultado: -1 = outlier, 1 = normal

# Convertir a etiqueta más legible
df['es_outlier'] = df['outlier_iforest'].map({-1: 'Outlier', 1: 'Normal'})

# --- 4. Resumen ---
n_outliers = (df['outlier_iforest'] == -1).sum()
n_total = len(df)
print(f"\n📊 Outliers detectados: {n_outliers} ({n_outliers/n_total*100:.1f}%)")
print(f"📊 Normales:            {n_total - n_outliers} ({(n_total-n_outliers)/n_total*100:.1f}%)")

📌 Variables usadas para Isolation Forest: ['PERCEPHO', 'P103', 'P104', 'P105A', 'EDUCACION_JEFE', 'DOMINIO', 'NUM_SERVICIOS', 'NUM_EQUIPOS', 'NUM_OCUPADOS', 'EDAD_JEFE', 'PCT_NINOS', 'HACINAMIENTO', 'outlier_iforest']

📊 Outliers detectados: 6571 (20.1%)
📊 Normales:            26193 (79.9%)


Mostrar distribución de los outliers por cluster

In [8]:
print("\n📋 Outliers por cluster (K-Means):")
tabla = pd.crosstab(
    df[METODO_PRINCIPAL],
    df['es_outlier'],
    margins=True
)
tabla['% Outlier'] = (tabla['Outlier'] / tabla['All'] * 100).round(1)
print(tabla)


📋 Outliers por cluster (K-Means):
es_outlier      Normal  Outlier    All  % Outlier
cluster_kmeans                                   
0                 5474      379   5853        6.5
1                 4059      272   4331        6.3
2                10316       44  10360        0.4
3                 6161      125   6286        2.0
4                 5115      819   5934       13.8
All              31125     1639  32764        5.0


# Cruzamos con DBSCAN
Los -1 de DBSCAN también son outliers — ¿coincidirán con Isolation Forest?

In [9]:
if 'cluster_dbscan' in df.columns:
    dbscan_outliers = df['cluster_dbscan'] == -1
    iforest_outliers = df['outlier_iforest'] == -1

    coincidencia = (dbscan_outliers & iforest_outliers).sum()
    print(f"\n🔁 Outliers detectados por AMBOS métodos: {coincidencia}")
    print(f"   Solo DBSCAN:          {(dbscan_outliers & ~iforest_outliers).sum()}")
    print(f"   Solo Isolation Forest: {(~dbscan_outliers & iforest_outliers).sum()}")


🔁 Outliers detectados por AMBOS métodos: 1639
   Solo DBSCAN:          31043
   Solo Isolation Forest: 0
